# Notebook 01: Bronze-Layer - Rohdaten-Ingestion

## Übersicht
Dieses Notebook implementiert die **Bronze-Layer-Komponente** der Medallion-Architektur. Es konsumiert Rohdaten aus dem Kafka-Topic und persistiert sie unverändert in einer Delta-Lake-Tabelle.

## Medallion Architecture - Bronze Layer
Der Bronze-Layer dient als **Raw Data Zone** mit folgenden Eigenschaften:
- ✅ Minimale Transformation
- ✅ Vollständige Rohdaten-Bewahrung
- ✅ Audit-Trail durch Ingestion-Metadaten
- ✅ Partitionierung für Performance

## Datenfluss
```
Kafka Topic (iot_tempdata)
    ↓
Streaming Consumer (PySpark)
    ↓
Metadaten hinzufügen (Timestamp, Offset, Partition)
    ↓
Bronze Delta Table (databricks_workspace.default.iot_data_bronze)
```

## Technische Details
- **Format**: Delta Lake (ACID-Transaktionen)
- **Partitionierung**: Nach `ingestion_date`
- **Checkpoint**: Fault-Tolerance aktiviert
- **Trigger**: availableNow (Serverless-kompatibel)

---

In [0]:
# BIBLIOTHEKEN UND ABHÄNGIGKEITEN

# PySpark Core
from pyspark.sql.functions import col, current_timestamp, to_date
from pyspark.sql.types import StructField, StructType, StringType, DoubleType

# Python Standard
from datetime import datetime

print(f"Notebook gestartet: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Alle Bibliotheken erfolgreich importiert
Notebook gestartet: 2026-03-10 13:29:13


In [0]:
# KONFIGURATION


# Kafka-Verbindungsparameter (Confluent Cloud)
KAFKA_BOOTSTRAP_SERVERS = "pkc-75m1o.europe-west3.gcp.confluent.cloud:9092"
KAFKA_TOPIC = "iot_tempdata"
KAFKA_USERNAME = "-------------"
KAFKA_PASSWORD = "-----------------------"

# Delta Lake Table-Konfiguration
BRONZE_TABLE = "databricks_workspace.default.iot_data_bronze"
CATALOG = "databricks_workspace"
SCHEMA = "default"
TABLE_NAME = "iot_data_bronze"

# Checkpoint-Pfad für Fault Tolerance
CHECKPOINT_PATH = "/Volumes/databricks_workspace/default/databricks/checkpoints/bronze_layer"

print("✓ Konfiguration geladen")
print(f"\n📊 BRONZE-LAYER-KONFIGURATION:")
print("="*60)
print(f"  Kafka Topic      : {KAFKA_TOPIC}")
print(f"  Bronze Table     : {BRONZE_TABLE}")
print(f"  Checkpoint-Pfad  : {CHECKPOINT_PATH}")
print(f"  Partitionierung  : ingestion_date")
print("="*60)

✓ Konfiguration geladen

📊 BRONZE-LAYER-KONFIGURATION:
  Kafka Topic      : iot_tempdata
  Bronze Table     : databricks_workspace.default.iot_data_bronze
  Checkpoint-Pfad  : /Volumes/databricks_workspace/default/databricks/checkpoints/bronze_layer
  Partitionierung  : ingestion_date


In [0]:
# KAFKA-CONSUMER ERSTELLEN


print("📡 Verbinde mit Kafka...\n")

# Kafka Streaming DataFrame erstellen
kafka_stream_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", 
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{KAFKA_USERNAME}" password="{KAFKA_PASSWORD}";')
    .option("startingOffsets", "earliest")  # Beginne am Anfang des Topics
    .load()
)

print("✓ Kafka-Consumer erfolgreich erstellt")
print(f"  - Topic: {KAFKA_TOPIC}")
print(f"  - Startposition: earliest")
print(f"  - Sicherheit: SASL_SSL")

# Zeige Kafka-Stream-Schema
print("\n📋 Kafka-Stream-Schema:")
kafka_stream_df.printSchema()

📡 Verbinde mit Kafka...

✓ Kafka-Consumer erfolgreich erstellt
  - Topic: iot_tempdata
  - Startposition: earliest
  - Sicherheit: SASL_SSL

📋 Kafka-Stream-Schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [0]:
# INGESTION-METADATEN HINZUFÜGEN


# Bronze-DataFrame mit Metadaten anreichern
bronze_df = (
    kafka_stream_df
    .selectExpr(
        "CAST(key AS STRING) as key",                    # Kafka-Key (device_id)
        "CAST(value AS STRING) as value",                # Kafka-Value (JSON payload)
        "topic",                                          # Kafka-Topic-Name
        "partition",                                      # Kafka-Partition
        "offset",                                         # Kafka-Offset
        "timestamp as kafka_timestamp"                    # Kafka-Timestamp
    )
    .withColumn("ingestion_timestamp", current_timestamp())   # Ingestion-Zeit
    .withColumn("ingestion_date", to_date(col("ingestion_timestamp")))  # Partition-Spalte
)

print("✓ Ingestion-Metadaten hinzugefügt")
print("\n📝 Bronze-Layer-Felder:")
print("  🔑 Kafka-Daten:")
print("     - key              : Device-ID (STRING)")
print("     - value            : JSON-Payload (STRING)")
print("     - topic            : Kafka-Topic")
print("     - partition        : Kafka-Partition")
print("     - offset           : Kafka-Offset")
print("     - kafka_timestamp  : Original-Kafka-Timestamp")
print("  ⏰ Ingestion-Metadaten:")
print("     - ingestion_timestamp : Zeitpunkt der Ingestion")
print("     - ingestion_date      : Partitionierungs-Datum")

# Zeige finales Schema
print("\n📋 Finales Bronze-Schema:")
bronze_df.printSchema()

✓ Ingestion-Metadaten hinzugefügt

📝 Bronze-Layer-Felder:
  🔑 Kafka-Daten:
     - key              : Device-ID (STRING)
     - value            : JSON-Payload (STRING)
     - topic            : Kafka-Topic
     - partition        : Kafka-Partition
     - offset           : Kafka-Offset
     - kafka_timestamp  : Original-Kafka-Timestamp
  ⏰ Ingestion-Metadaten:
     - ingestion_timestamp : Zeitpunkt der Ingestion
     - ingestion_date      : Partitionierungs-Datum

📋 Finales Bronze-Schema:
root
 |-- key: string (nullable = true)
 |-- value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = false)



In [0]:
# BRONZE DELTA TABLE SCHREIBEN


print("🚀 Starte Bronze-Layer-Ingestion...\n")

# Streaming-Query zur Bronze-Tabelle
bronze_query = (
    bronze_df
    .writeStream
    .format("delta")                               # Delta Lake Format
    .outputMode("append")                          # Nur neue Daten anfügen
    .option("checkpointLocation", CHECKPOINT_PATH) # Checkpoint für Fault-Tolerance
    .partitionBy("ingestion_date")                 # Partitionierung nach Datum
    .trigger(availableNow=True)                    # Serverless-kompatibel
    .toTable(BRONZE_TABLE)                         # Ziel-Tabelle
)

print("⏳ Warte auf Streaming-Abschluss...")

# Warte auf Abschluss
bronze_query.awaitTermination()

print("\n" + "="*60)
print("✓ BRONZE-LAYER-INGESTION ERFOLGREICH ABGESCHLOSSEN")
print("="*60)
print(f"  ✓ Tabelle: {BRONZE_TABLE}")
print(f"  ✓ Format: Delta Lake")
print(f"  ✓ Partitionierung: ingestion_date")
print(f"  ✓ Checkpoint: {CHECKPOINT_PATH}")
print("="*60)

🚀 Starte Bronze-Layer-Ingestion...

⏳ Warte auf Streaming-Abschluss...


26/03/10 13:57:07 Query 14abdd3c-58f0-4394-ba3c-140b12b2b880 have exception: org.apache.spark.SparkException: Exception thrown in awaitResult: Some data may have been lost because they are not available in Kafka any more;
either the data was aged out by Kafka or the topic may have been deleted before all the data in the
topic was processed.
If you don't want your streaming query to fail on such cases, set the source option failOnDataLoss to false.
Reason: Could not read records in offset [115336, 433379) for topic partition iot_tempdata-1
with consumer group spark-kafka-source-96fac3af-6f91-4f1e-b5fe-c8f428e9212b-1536636248-executor.
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:51)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:556)
	at com.databricks.sql.transaction.tahoe.perf.DeltaOptimizedWriterExec.awaitShuffleMapStage$1(DeltaOptimizedWriterExec.scala:204)
	at com.databricks.sql.transaction.tahoe.perf.DeltaOptimizedWriterExec.

---------------------------------------------------------------------------
StreamingQueryException                   Traceback (most recent call last)
File <command-5784020168237836>, line 22
     19 print("⏳ Warte auf Streaming-Abschluss...")
     21 # Warte auf Abschluss
---> 22 bronze_query.awaitTermination()
     24 print("\n" + "="*60)
     25 print("✓ BRONZE-LAYER-INGESTION ERFOLGREICH ABGESCHLOSSEN")

File /databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/streaming/query.py:102, in StreamingQuery.awaitTermination(self, timeout)
    100 await_termination_cmd = pb2.StreamingQueryCommand.AwaitTerminationCommand()
    101 cmd.await_termination.CopyFrom(await_termination_cmd)
--> 102 self._execute_streaming_query_cmd(cmd)
    103 return None

File /databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/streaming/query.py:211, in StreamingQuery._execute_streaming_query_cmd(self, cmd)
    209 exec_cmd = pb2.Command()
    210 exec_cmd.streaming_query_comma

In [0]:
# BRONZE-TABELLE VERIFIZIEREN


print("⚙️ Überprüfe Bronze-Tabelle...\n")

# 1. Tabellen-Existenz prüfen
try:
    bronze_table_df = spark.read.format("delta").table(BRONZE_TABLE)
    print("✓ Tabelle existiert und ist lesbar")
except Exception as e:
    print(f"❌ Fehler beim Lesen der Tabelle: {e}")
    raise

# 2. Anzahl Datensätze
total_records = bronze_table_df.count()
print(f"\n📊 BRONZE-LAYER-STATISTIKEN:")
print("="*60)
print(f"  Gesamtanzahl Datensätze : {total_records:,}")

# 3. Partitions-Übersicht
partitions = bronze_table_df.select("ingestion_date").distinct().count()
print(f"  Anzahl Partitionen       : {partitions}")

# 4. Schema-Übersicht
print(f"  Anzahl Spalten           : {len(bronze_table_df.columns)}")
print(f"  Spalten                  : {', '.join(bronze_table_df.columns)}")

# 5. Zeitraum der Daten
if total_records > 0:
    from pyspark.sql.functions import min as spark_min, max as spark_max
    time_range = bronze_table_df.agg(
        spark_min("kafka_timestamp").alias("min_ts"),
        spark_max("kafka_timestamp").alias("max_ts")
    ).collect()[0]
    print(f"\n⏰ ZEITRAUM:")
    print(f"  Frühester Timestamp     : {time_range['min_ts']}")
    print(f"  Spätester Timestamp      : {time_range['max_ts']}")

print("="*60)

In [0]:
# STICHPROBEN-PRÜFUNG


print("🔍 Zeige Beispieldaten aus Bronze-Layer:\n")

# Zeige die neuesten 5 Einträge
latest_records = (
    spark.read.format("delta").table(BRONZE_TABLE)
    .orderBy(col("kafka_timestamp").desc())
    .limit(5)
)

print("📄 Neueste 5 Datensätze:")
print("="*60)
display(latest_records)

In [0]:
# TABELLEN-DETAILS


print("📊 BRONZE-TABELLEN-DETAILS:\n")

# DESCRIBE DETAIL für Delta-Tabelle
table_details = spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}").collect()[0]

print("="*60)
print("DELTA LAKE EIGENSCHAFTEN:")
print("="*60)
print(f"  Tabellen-ID          : {table_details['id']}")
print(f"  Format               : {table_details['format']}")
print(f"  Speicherort          : {table_details['location']}")
print(f"  Anzahl Dateien       : {table_details['numFiles']}")
print(f"  Größe (MB)           : {table_details['sizeInBytes'] / (1024*1024):.2f}")
print(f"  Partitions-Spalten   : {table_details['partitionColumns']}")
print("="*60)

# Zeige Tabellen-Schema
print("\n📋 TABELLEN-SCHEMA:")
spark.sql(f"DESCRIBE {BRONZE_TABLE}").show(truncate=False)

In [0]:
# ZUSAMMENFASSUNG UND NÄCHSTE SCHRITTE


print("\n" + "🎉 "*30)
print("\n📊 BRONZE-LAYER-ZUSAMMENFASSUNG")
print("\n" + "="*60)

# Finale Statistiken
final_count = spark.read.format("delta").table(BRONZE_TABLE).count()

print(f"\n1️⃣ DATEN-INGESTION:")
print(f"   ✓ Datensätze verarbeitet  : {final_count:,}")
print(f"   ✓ Quelle                 : Kafka Topic '{KAFKA_TOPIC}'")
print(f"   ✓ Ziel                   : Delta Table '{BRONZE_TABLE}'")

print(f"\n2️⃣ QUALITÄTSMERKMALE:")
print(f"   ✓ Rohdaten-Bewahrung     : 100% (keine Transformation)")
print(f"   ✓ Audit-Trail            : Vollständig (Kafka-Metadaten)")
print(f"   ✓ Fault-Tolerance        : Aktiviert (Checkpoints)")
print(f"   ✓ ACID-Transaktionen     : Delta Lake")

print(f"\n3️⃣ PERFORMANCE:")
print(f"   ✓ Partitionierung        : Nach ingestion_date")
print(f"   ✓ Format                 : Delta Lake (Parquet + Log)")
print(f"   ✓ Checkpoint-basiert     : Ja")

print(f"\n4️⃣ NÄCHSTE SCHRITTE:")
print(f"   → Notebook 03: Silver-Layer-Validation ausführen")
print(f"   → Datenqualitäts-Prüfungen anwenden")
print(f"   → Validierte Daten in Silver-Layer überführen")

print("\n" + "="*60)
print("✓ NOTEBOOK 02 ERFOLGREICH AUSGEFÜHRT")
print("="*60)
print("\n" + "🎉 "*30)